# Project 51 — Local Web Research Agent
## Search, Compare Sources, Write Cited Answers

**Stack:** LangChain · Ollama · DuckDuckGo · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community duckduckgo-search

## Step 1 — Setup with Tool Definitions

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

@tool
def web_search(query: str) -> str:
    """Search the web for information. Returns top results."""
    # Simulated search results for offline use
    results = {
        "RAG vs fine-tuning": [
            "RAG retrieves documents at inference; fine-tuning modifies model weights. RAG is cheaper to update.",
            "Fine-tuning excels at style/tone adaptation; RAG excels at factual accuracy with citations.",
        ],
        "Ollama models 2025": [
            "Ollama supports Llama 3, Mistral, Qwen, Phi-3, Gemma and more for local execution.",
            "Ollama 0.5 introduced multi-model serving and 30% memory reduction.",
        ],
    }
    for key, vals in results.items():
        if any(w in query.lower() for w in key.lower().split()):
            return "\n".join(vals)
    return "No specific results found. Try a more specific query."

@tool
def summarize_text(text: str) -> str:
    """Summarize a long text into key points."""
    chain = ChatPromptTemplate.from_messages([
        ("system", "Summarize in 3 bullet points."),
        ("human", "{text}")
    ]) | llm | StrOutputParser()
    return chain.invoke({"text": text})

tools = [web_search, summarize_text]
print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

## Step 2 — Build Research Agent

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research agent with access to web search and summarization tools.
For each question:
1. Search for relevant information
2. Summarize findings
3. Write a cited answer

Always cite your sources."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# For models that don't support tool calling natively, use ReAct
from langchain.agents import create_react_agent

react_prompt = ChatPromptTemplate.from_template(
    """Answer the question using the available tools.

Tools: {tools}
Tool Names: {tool_names}

Question: {input}

Think step by step.
{agent_scratchpad}"""
)

# Direct approach without full agent (works reliably with local models)
def research(question):
    search_result = web_search.invoke(question)
    summary = summarize_text.invoke(search_result)
    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "Write a comprehensive answer using the research. Cite sources."),
        ("human", "Question: {question}\nResearch: {research}\nSummary: {summary}")
    ])
    chain = answer_prompt | llm | StrOutputParser()
    return chain.invoke({"question": question, "research": search_result, "summary": summary})

print("Research agent ready!")

## Step 3 — Run Research Queries

In [ ]:
queries = [
    "What's the difference between RAG and fine-tuning?",
    "What models does Ollama support in 2025?",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Research Q: {q}")
    print("="*60)
    answer = research(q)
    print(answer)

## What You Learned
- **Tool-using agents** with search and summarization
- **Research pipeline:** search → summarize → cite → answer
- **Offline-friendly** design with simulated search results